In [1]:
import torch
from ts_url.training_methods import Trainer
import time
import os 

/home/liangchen/miniconda3/envs/UniTS/lib/python3.8/site-packages/tslearn/bases/bases.py:15: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


Try csl on LSST dataset with SVM as the test module.

In [9]:
import json 
from ts_url.models.default_configs.configues import optim_configures, task_configures, model_configures

experiment = "exp1"

hp_path = "configs/csl_optim.json"
p_path = "configs/csl.json"
optim_config = "configs/csl_optim.json"
task_name = "pretraining"
model_name = "csl"


with open(optim_config, 'r') as f:
    optim_config = json.load(f)
    
optim_config["evaluator"] = "svm"


print(optim_config)


device = torch.device('cuda')


def get_config(filepath="", train_ratio=1, test_ratio=1, dsid="CarVibration1"):
    data_configs = [{
        "filepath": filepath,
        "train_ratio": train_ratio,
        "test_ratio": test_ratio,
        "dsid": dsid
    }]
    return data_configs

data_configs = get_config()

data_names = [d['dsid'] for d in data_configs]
task_summary = "_".join(data_names) + "_" + model_name
start_time = time.strftime("%m_%d_%H_%M_%S", time.localtime()) 


# task_summary = "_".join(task["data_name"]) + "_" + model_name
save_name = start_time + "_" + task_summary
  
save_path = os.path.join(experiment, task_summary, save_name)

os.makedirs(save_path, exist_ok=True)

import random
import numpy as np
random.seed(0)
torch.manual_seed(0)
np.random.seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True
trainer = Trainer(data_configs, model_name, p_path, 
                  device, optim_config, task_name, save_path=save_path)

ckpt = ["/home/liangchen/UniTS/exp1/CarVibration1_csl/03_05_22_58_23_CarVibration1_csl"]

# fine_tune_config = {"fusion":"concat"}
task="classification"
if task != "pretraining":
    with open(task_configures[task], "r") as oc:
        optim_config = json.load(oc)
    ckpt = ckpt
    fusion = "concat"
    fine_tune_config = {"fusion": fusion}
    # if task_name == "regression":
    #     fine_tune_config["pred_len"] = task["pred_len"]
    # if task_name == "imputation":
    #     fine_tune_config["i_ratio"] = task["i_ratio"]
        
        
trainer = Trainer(data_configs, model_name, p_path, 
					device=device, task=task, optim_config=optim_config, fine_tune_config=fine_tune_config, ckpt_paths=ckpt)

trainer.fit()



2025-03-06 00:01:53,046 | INFO : train_ds length: 40, valid_ds length: 40
2025-03-06 00:01:53,059 | INFO : {'output_dims': 320, 'feat_dim': 6, 'max_len': 100, 'device': 'cpu'}
2025-03-06 00:01:53,060 | INFO : CSL(
  (shapelets_euclidean): ShapeletsDistBlocks(
    (blocks): ModuleList(
      (0-7): 8 x MinEuclideanDistBlock()
    )
  )
  (shapelets_cosine): ShapeletsDistBlocks(
    (blocks): ModuleList(
      (0-7): 8 x MaxCosineSimilarityBlock(
        (relu): ReLU()
      )
    )
  )
  (shapelets_cross_correlation): ShapeletsDistBlocks(
    (blocks): ModuleList(
      (0): MaxCrossCorrelationBlock(
        (shapelets): Conv1d(6, 14, kernel_size=(10,), stride=(1,))
      )
      (1): MaxCrossCorrelationBlock(
        (shapelets): Conv1d(6, 14, kernel_size=(20,), stride=(1,))
      )
      (2): MaxCrossCorrelationBlock(
        (shapelets): Conv1d(6, 14, kernel_size=(30,), stride=(1,))
      )
      (3): MaxCrossCorrelationBlock(
        (shapelets): Conv1d(6, 14, kernel_size=(40,), str

2025-03-06 00:01:53,114 | INFO : train_ds length: 40, valid_ds length: 40


{'T': 0.1, 'alpha': 0.5, 'l3': 0.01, 'l4': 1.0, 'mean_mask_length': 3, 'masking_ratio': 0.15, 'mask_mode': 'separate', '@mask_mode/choice': ['seperate', 'concurrent'], 'mask_distribution': 'geometric', '@mask_distribution/choice': ['geometric', 'bernoulli'], 'exclude_feats': None, 'batch_size': 8, 'optimizer': 'SGD', '@optimier/choice': ['Adam', 'RAdam'], 'lr': 0.01, 'l2_reg': 0, '@epochs': 10, 'epochs': 10, 'print_interval': 10, 'evaluator': 'svm'}
{'T': 0.1, 'alpha': 0.5, 'l3': 0.01, 'l4': 1.0, 'mean_mask_length': 3, 'masking_ratio': 0.15, 'mask_mode': 'separate', '@mask_mode/choice': ['seperate', 'concurrent'], 'mask_distribution': 'geometric', '@mask_distribution/choice': ['geometric', 'bernoulli'], 'exclude_feats': None, 'batch_size': 8, 'optimizer': 'SGD', '@optimier/choice': ['Adam', 'RAdam'], 'lr': 0.01, 'l2_reg': 0, '@epochs': 10, 'epochs': 10, 'print_interval': 10, 'evaluator': 'svm'}
csl
6 None
[{'output_dims': 320, 'feat_dim': 6, 'max_len': 100, 'device': device(type='cuda'

/home/liangchen/UniTS/ts_url/process_model.py:179: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  getattr(self, "model_" + str(idx)).load_state_dict(torch.load(ckpt_path)["st

Training Epoch 0   0.0% | batch:         0 of         1	|	loss: 1.90582

...

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10
           2       1.00      1.00      1.00        10
           3       1.00      1.00      1.00        10

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

loss
[{'output_dims': 320, 'feat_dim': 6, 'max_len': 100, 'device': 'cuda'}]
Training Epoch 99   0.0% | batch:         0 of         1	|	loss: 0.00157969

2025-03-06 00:02:32,311 | INFO : Evaluating on validation set ...


Evaluating Epoch 99  75.0% | batch:        30 of        40	|	loss: 0.02235993

2025-03-06 00:02:32,657 | INFO : Validation runtime: 0.0 hours, 0.0 minutes, 0.3446464538574219 seconds

2025-03-06 00:02:32,659 | INFO : Avg val. time: 0.0 hours, 0.0 minutes, 0.3376809859275818 seconds
2025-03-06 00:02:32,661 | INFO : Avg batch val. time: 0.008442024648189545 seconds
2025-03-06 00:02:32,662 | INFO : Avg sample val. time: 0.008442024648189545 seconds
2025-03-06 00:02:32,664 | INFO : Epoch 99 Validation Summary: epoch: 99.000000 | loss: 0.070143 | 


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10
           2       1.00      1.00      1.00        10
           3       1.00      1.00      1.00        10

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

loss
[{'output_dims': 320, 'feat_dim': 6, 'max_len': 100, 'device': 'cuda'}]


In [3]:
import json
from ts_url.models.default_configs.configues import optim_configures, task_configures, model_configures


experiment = "exp1"

hp_path = "configs/csl_optim.json"
p_path = "configs/csl.json"
optim_config = "configs/csl_optim.json"
task_name = "pretraining"
model_name = "csl"

with open(optim_config, 'r') as f:
    optim_config = json.load(f)
    
optim_config["evaluator"] = "ridge"

device = torch.device('cuda')


def get_config(filepath="", train_ratio=1, test_ratio=1, dsid="CarVibration1"):
    data_configs = [{
        "filepath": filepath,
        "train_ratio": train_ratio,
        "test_ratio": test_ratio,
        "dsid": dsid
    }]
    return data_configs

data_configs = get_config()

data_names = [d['dsid'] for d in data_configs]
task_summary = "_".join(data_names) + "_" + model_name
start_time = time.strftime("%m_%d_%H_%M_%S", time.localtime()) 


# task_summary = "_".join(task["data_name"]) + "_" + model_name
save_name = start_time + "_" + task_summary
  
save_path = os.path.join(experiment, task_summary, save_name)

os.makedirs(save_path, exist_ok=True)


import random
import numpy as np
random.seed(0)
torch.manual_seed(0)
np.random.seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True
trainer = Trainer(data_configs, model_name, p_path, 
                  device, optim_config, task_name, save_path=save_path)

ckpt = ["/home/liangchen/UniTS/exp1/CarVibration1_csl/03_06_00_09_07_CarVibration1_csl"]

task="regression"
if task != "pretraining":
    with open(task_configures[task], "r") as oc:
        optim_config = json.load(oc)
    ckpt = ckpt
    fusion = "concat"
    fine_tune_config = {"fusion": fusion, "pred_len": 3}
print(optim_config)

# fine_tune_config = {"fusion":"concat"}
trainer = Trainer(data_configs, model_name, p_path, 
					device=device, task=task, optim_config=optim_config, fine_tune_config=fine_tune_config, ckpt_paths=ckpt)

trainer.fit()

2025-03-06 00:18:43,300 | INFO : train_ds length: 40, valid_ds length: 40
2025-03-06 00:18:43,313 | INFO : {'output_dims': 320, 'feat_dim': 6, 'max_len': 100, 'device': 'cpu'}
2025-03-06 00:18:43,314 | INFO : CSL(
  (shapelets_euclidean): ShapeletsDistBlocks(
    (blocks): ModuleList(
      (0-7): 8 x MinEuclideanDistBlock()
    )
  )
  (shapelets_cosine): ShapeletsDistBlocks(
    (blocks): ModuleList(
      (0-7): 8 x MaxCosineSimilarityBlock(
        (relu): ReLU()
      )
    )
  )
  (shapelets_cross_correlation): ShapeletsDistBlocks(
    (blocks): ModuleList(
      (0): MaxCrossCorrelationBlock(
        (shapelets): Conv1d(6, 14, kernel_size=(10,), stride=(1,))
      )
      (1): MaxCrossCorrelationBlock(
        (shapelets): Conv1d(6, 14, kernel_size=(20,), stride=(1,))
      )
      (2): MaxCrossCorrelationBlock(
        (shapelets): Conv1d(6, 14, kernel_size=(30,), stride=(1,))
      )
      (3): MaxCrossCorrelationBlock(
        (shapelets): Conv1d(6, 14, kernel_size=(40,), str

{'T': 0.1, 'alpha': 0.5, 'l3': 0.01, 'l4': 1.0, 'mean_mask_length': 3, 'masking_ratio': 0.15, 'mask_mode': 'separate', '@mask_mode/choice': ['seperate', 'concurrent'], 'mask_distribution': 'geometric', '@mask_distribution/choice': ['geometric', 'bernoulli'], 'exclude_feats': None, 'batch_size': 8, 'optimizer': 'SGD', '@optimier/choice': ['Adam', 'RAdam'], 'lr': 0.01, 'l2_reg': 0, '@epochs': 10, 'epochs': 10, 'print_interval': 10, 'evaluator': 'ridge'}
{'batch_size': 64, 'optimizer': 'Adam', '@optimier/choice': ['Adam', 'RAdam'], 'lr': 0.001, 'l2_reg': 0, 'print_interval': 10, 'epochs': 100}
csl
6 3
[{'output_dims': 320, 'feat_dim': 6, 'max_len': 100, 'device': device(type='cuda')}]


/home/liangchen/UniTS/ts_url/process_model.py:179: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  getattr(self, "model_" + str(idx)).load_state_dict(torch.load(ckpt_path)["st

Training Epoch 0   0.0% | batch:         0 of         1	|	loss: 1.17598

2025-03-06 00:18:45,525 | INFO : Evaluating on validation set ...


...

1.01915475204587
loss
Training Epoch 99   0.0% | batch:         0 of         1	|	loss: 5.5638e-06

2025-03-06 00:19:21,908 | INFO : Evaluating on validation set ...


Evaluating Epoch 99  75.0% | batch:        30 of        40	|	loss: 0.351113

2025-03-06 00:19:22,237 | INFO : Validation runtime: 0.0 hours, 0.0 minutes, 0.32770681381225586 seconds

2025-03-06 00:19:22,238 | INFO : Avg val. time: 0.0 hours, 0.0 minutes, 0.32853755712509153 seconds
2025-03-06 00:19:22,239 | INFO : Avg batch val. time: 0.008213438928127289 seconds
2025-03-06 00:19:22,241 | INFO : Avg sample val. time: 0.008213438928127289 seconds
2025-03-06 00:19:22,242 | INFO : Epoch 99 Validation Summary: epoch: 99.000000 | loss: 1.019197 | 


1.0191969852894545
loss


Try ts_tcc on LSST dataset with SVM as test module.

In [3]:
experiment = "exp1"

# hp_path = "configs/ts_tcc_optim.json"
p_path = "configs/ts_tcc.json"
optim_config = "configs/ts_tcc_optim.json"
task_name = "pretraining"
model_name = "ts_tcc"

device = torch.device('cuda')

data_names = [d['dsid'] for d in data_configs]
task_summary = "_".join(data_names) + "_" + model_name
start_time = time.strftime("%m_%d_%H_%M_%S", time.localtime()) 
save_path = os.path.join(experiment, task_summary, start_time)

os.makedirs(save_path, exist_ok=True)

Try t_loss on LSST dataset with SVM as test module.

In [ ]:
experiment = "exp1"

# hp_path = "configs/ts_tcc_optim.json"
p_path = "configs/t_loss.json"
optim_config = "configs/t_loss_optim.json"
task_name = "pretraining"
model_name = "t_loss"

device = torch.device('cuda')

data_names = [d['dsid'] for d in data_configs]
task_summary = "_".join(data_names) + "_" + model_name
start_time = time.strftime("%m_%d_%H_%M_%S", time.localtime()) 
save_path = os.path.join(experiment, task_summary, start_time)

os.makedirs(save_path, exist_ok=True)

In [3]:
experiment = "exp1"

# hp_path = "configs/ts_tcc_optim.json"
p_path = "ts_url/models/default_configs/mvts_transformer.json"
optim_config = "ts_url/models/default_configs/mvts_transformer_optim.json"
task_name = "pretraining"
model_name = "mvts_transformer"

device = torch.device('cuda')

data_names = [d['dsid'] for d in data_configs]
task_summary = "_".join(data_names) + "_" + model_name
start_time = time.strftime("%m_%d_%H_%M_%S", time.localtime()) 
save_path = os.path.join(experiment, task_summary, start_time)

os.makedirs(save_path, exist_ok=True)

In [3]:
experiment = "exp1"

# hp_path = "configs/ts_tcc_optim.json"
p_path = "ts_url/models/default_configs/ts2vec.json"
optim_config = "ts_url/models/default_configs/ts2vec_optim.json"
task_name = "pretraining"
model_name = "ts2vec"

device = torch.device('cuda')

data_names = [d['dsid'] for d in data_configs]
task_summary = "_".join(data_names) + "_" + model_name
start_time = time.strftime("%m_%d_%H_%M_%S", time.localtime()) 
save_path = os.path.join(experiment, task_summary, start_time)

os.makedirs(save_path, exist_ok=True)

In [3]:


import random
import numpy as np
random.seed(0)
torch.manual_seed(0)
np.random.seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True
trainer = Trainer(data_configs, model_name, p_path, 
                  device, optim_config, task_name, save_path=save_path)

trainer.fit()


/home/liangchen/liangchen/UniTS/ts_url/process_data.py:503: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1670525552411/work/torch/csrc/utils/tensor_new.cpp:230.)
  self.data = torch.tensor(data)  # this is a subclass of the BaseData class in data.py
2024-02-12 22:39:40,919 | INFO : train_ds length: 2459, valid_ds length: 2466
2024-02-12 22:39:40,927 | INFO : {'output_dims': 320, 'feat_dim': 6, 'max_len': 36, 'device': 'cpu'}
2024-02-12 22:39:40,928 | INFO : CSL(
  (shapelets_euclidean): ShapeletsDistBlocks(
    (blocks): ModuleList(
      (0): MinEuclideanDistBlock()
      (1): MinEuclideanDistBlock()
      (2): MinEuclideanDistBlock()
      (3): MinEuclideanDistBlock()
      (4): MinEuclideanDistBlock()
      (5): MinEuclideanDistBlock()
      (6): MinEuclideanDistBlock()
      (7): MinE

Training Epoch 0  97.4% | batch:       300 of       308	|	loss: 7.359742

2024-02-12 22:41:04,291 | INFO : Evaluating on validation set ...


Evaluating Epoch 0   1.2% | batch:        30 of      2466	|	loss: inf

/home/liangchen/liangchen/UniTS/ts_url/process_data.py:530: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X).transpose(0, 1)


...

/home/liangchen/liangchen/UniTS/ts_url/process_data.py:530: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X).transpose(0, 1)


Evaluating Epoch 1  99.8% | batch:      2460 of      2466	|	loss: inf

/home/liangchen/miniconda3/envs/UniTS/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/liangchen/miniconda3/envs/UniTS/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/liangchen/miniconda3/envs/UniTS/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
2024-02-12

              precision    recall  f1-score   support

           0       0.06      0.50      0.10         4
           1       0.40      0.59      0.48        85
           2       0.86      0.76      0.81       307
           3       0.40      0.40      0.40       386
           4       0.02      0.08      0.03        12
           5       0.00      0.00      0.00         0
           6       0.13      0.25      0.17        80
           7       0.00      0.00      0.00         0
           8       0.84      0.69      0.76       382
           9       0.04      0.38      0.08         8
          10       0.88      0.96      0.92       112
          11       0.80      0.60      0.69      1028
          12       0.39      0.60      0.47        50
          13       0.15      0.67      0.25        12

    accuracy                           0.60      2466
   macro avg       0.36      0.46      0.37      2466
weighted avg       0.70      0.60      0.64      2466

loss
Training Epoch 2  97

2024-02-12 22:46:17,336 | INFO : Evaluating on validation set ...


In [5]:
from ts_url.training_methods import Trainer
from ts_url.registry import * 
print(list(EVALUATE._registry.keys()))

AttributeError: 'dict' object has no attribute '__dict__'